# TrustShift-IDS WP2 — V8: exact frozen primary-monitor replay

**Source of truth:** repository `notebooks/02_TrustShift_V2_frozen_pipeline.ipynb`, original **V2.8C-R10** code cell. The core below is copied without methodological changes. It recomputes all 82 clean-IID windows from frozen models, calibration data and feature mapping, and checks against the historical regression file.

**Scope:** Phase A historical XGBoost exactness, followed by Phase B neural-readiness checks. **This notebook does not claim neural detector-specific evaluation or final WP2 outcomes.** No deployment labels are used for monitor calibration. Run in Google Colab with Drive mounted; GPU not necessary. Computing 82 × 68 Wasserstein comparisons against 274,787 reference rows may take substantial CPU time.

**Stop rule:** If the original R10 gate fails, subsequent cells must not run. No existing original files are overwritten.

In [ ]:
# A0 — mount and environment
%pip -q install pandas numpy scipy pyarrow joblib scikit-learn xgboost
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
from pathlib import Path
WP2_OUT=Path('/content/drive/MyDrive/TrustShift_IDS/V2_adaptive_reliability/WP2_detector_specific')
WP2_OUT.mkdir(parents=True,exist_ok=True)
print('Output folder:',WP2_OUT)

## A1 — Original V2.8C-R10 exact replay

The following is the **unmodified original scientific computation and regression gate**, extracted from the repository. It loads source-only calibration signals and the clean IID test for regression verification, not deployment evaluation. Expected: 82 windows, state counts 75/5/2, maximum SSS difference ≤1e−9.

In [ ]:
# ==========================================================================================
# TRUSTSHIFT-IDS V2
# STEP V2.8C-R10 — FINAL EXACT CLEAN-IID PRIMARY-MONITOR REGRESSION GATE
#
# Recovered directly from original V2.4A / V2.6E-4 implementation.
#
# CLEAN IID ONLY.
# NO S5.
# NO labels/outcomes.
# NO adaptive policy.
# NO monitor modification.
# ==========================================================================================

import os
import numpy as np
import pandas as pd
import joblib

from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon

BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2   = f"{BASE}/V2_adaptive_reliability"

SRC = f"{V2}/source_reference"
MON = f"{V2}/shift_monitoring"
AUD = f"{V2}/artifact_audits"

WINDOW = 5000
SEED = 42
EPS = 1e-12

T_BINARY = 0.9744155346869672
T_MULTI  = 1.1709901909513676

Q95 = 0.4413449708140494
Q99 = 0.6791419174145775

IID_PATH = (
    f"{BASE}/data/processed/CICIDS2017_68F_imputed/"
    "CICIDS2017_iid_test_68F_imputed.parquet"
)

CAL_PATH = (
    f"{BASE}/data/processed/CICIDS2017_68F_imputed/"
    "CICIDS2017_calibration_68F_imputed.parquet"
)

B_MODEL_PATH = (
    f"{BASE}/models/xgboost/"
    "XGB_binary_seed42.joblib"
)

M_MODEL_PATH = (
    f"{BASE}/models/xgboost/"
    "XGB_multiclass_seed42.joblib"
)

FEATURE_MAP_PATH = (
    f"{SRC}/V2_frozen_68_feature_mapping.csv"
)

SIGNAL_REF_PATH = (
    f"{SRC}/V2_calibration_label_free_signals.parquet"
)

NORM_PATH = (
    f"{MON}/V2_monitoring_normalization_parameters.csv"
)

FROZEN_PATH = (
    f"{AUD}/V2_STEP_6E4_REPRODUCED_CLEAN_IID_MONITORING.csv"
)

print("=" * 108)
print("V2.8C-R10 — FINAL EXACT CLEAN-IID PRIMARY-MONITOR REGRESSION GATE")
print("=" * 108)


# ==========================================================================================
# 1. LOAD FROZEN INPUTS
# ==========================================================================================

for path in [
    IID_PATH,
    CAL_PATH,
    B_MODEL_PATH,
    M_MODEL_PATH,
    FEATURE_MAP_PATH,
    SIGNAL_REF_PATH,
    NORM_PATH,
    FROZEN_PATH,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)


mapping = pd.read_csv(FEATURE_MAP_PATH)

if "feature_id" in mapping.columns:
    FEATURES = mapping["feature_id"].tolist()
elif "feature" in mapping.columns:
    FEATURES = mapping["feature"].tolist()
else:
    raise RuntimeError(
        f"Cannot identify feature column: {mapping.columns.tolist()}"
    )

assert len(FEATURES) == 68

Xcal = pd.read_parquet(
    CAL_PATH,
    columns=FEATURES
).to_numpy(dtype=np.float64)

Xiid = pd.read_parquet(
    IID_PATH,
    columns=FEATURES
).to_numpy(dtype=np.float64)

assert Xcal.shape == (274787, 68)
assert Xiid.shape == (412181, 68)

b_model = joblib.load(B_MODEL_PATH)
m_model = joblib.load(M_MODEL_PATH)

ref_signals = pd.read_parquet(SIGNAL_REF_PATH)

norm = pd.read_csv(NORM_PATH)

if "indicator" in norm.columns:
    norm = norm.set_index("indicator")

frozen = pd.read_csv(FROZEN_PATH)

assert len(frozen) == 82

print("\n[1] Frozen inputs: PASS")


# ==========================================================================================
# 2. EXACT FROZEN HELPERS
# ==========================================================================================

def temp_scale(p, T):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    p = np.clip(
        p,
        1e-15,
        1.0
    )

    z = np.log(p) / T
    z -= np.max(z, axis=1, keepdims=True)

    ez = np.exp(z)

    return (
        ez
        /
        np.sum(ez, axis=1, keepdims=True)
    )


def standardized_wasserstein(reference, window):

    reference = np.asarray(
        reference,
        dtype=np.float64
    )

    window = np.asarray(
        window,
        dtype=np.float64
    )

    q25, q75 = np.quantile(
        reference,
        [0.25, 0.75]
    )

    scale = q75 - q25

    # EXACT frozen fallback.
    if (
        not np.isfinite(scale)
        or scale < EPS
    ):
        scale = np.std(reference)

    if (
        not np.isfinite(scale)
        or scale < EPS
    ):
        return 0.0

    return float(
        wasserstein_distance(
            reference,
            window
        )
        /
        scale
    )


def js_divergence(reference, window, classes):

    rc = np.array(
        [
            np.sum(reference == c)
            for c in classes
        ],
        dtype=np.float64
    )

    wc = np.array(
        [
            np.sum(window == c)
            for c in classes
        ],
        dtype=np.float64
    )

    rc += 1e-12
    wc += 1e-12

    p = rc / rc.sum()
    q = wc / wc.sum()

    # scipy returns JS distance.
    # Frozen implementation squares it to obtain divergence.
    return float(
        jensenshannon(
            p,
            q,
            base=2
        ) ** 2
    )


def normalize_indicator(name, value):

    row = norm.loc[name]

    mode = row["normalization_mode"]

    if mode == "structural_zero_null":
        return float(value > 0)

    med = float(
        row["null_median"]
    )

    scale = float(
        row["scale_median_to_q95"]
    )

    if (
        not np.isfinite(scale)
        or scale <= 0
    ):
        return 0.0

    return max(
        0.0,
        (float(value) - med) / scale
    )


print("[2] Exact frozen helpers: PASS")


# ==========================================================================================
# 3. IID LABEL-FREE SIGNALS
# ==========================================================================================

print("\n[3] Computing IID label-free predictive signals...")

pb = temp_scale(
    b_model.predict_proba(Xiid),
    T_BINARY
)

pm = temp_scale(
    m_model.predict_proba(Xiid),
    T_MULTI
)

iid_signals = pd.DataFrame({
    "binary_confidence":
        np.max(pb, axis=1),

    "binary_entropy":
        -np.sum(
            pb * np.log(
                np.clip(pb, 1e-15, 1.0)
            ),
            axis=1
        ) / np.log(2),

    "binary_margin":
        np.sort(pb, axis=1)[:, -1]
        -
        np.sort(pb, axis=1)[:, -2],

    "binary_predicted_class":
        np.argmax(pb, axis=1),

    "multiclass_confidence":
        np.max(pm, axis=1),

    "multiclass_entropy":
        -np.sum(
            pm * np.log(
                np.clip(pm, 1e-15, 1.0)
            ),
            axis=1
        ) / np.log(9),

    "multiclass_margin":
        np.sort(pm, axis=1)[:, -1]
        -
        np.sort(pm, axis=1)[:, -2],

    "multiclass_predicted_class":
        np.argmax(pm, axis=1),
})

del pb, pm

print("IID predictive signals: PASS")


# ==========================================================================================
# 4. EXACT SEED-42 WINDOW MEMBERSHIP
# ==========================================================================================

perm = np.random.default_rng(
    SEED
).permutation(len(Xiid))

N_WINDOWS = len(Xiid) // WINDOW

windows = [
    perm[
        i * WINDOW:
        (i + 1) * WINDOW
    ]
    for i in range(N_WINDOWS)
]

assert N_WINDOWS == 82

print("\n[4] Window protocol: PASS")
print("Windows    :", N_WINDOWS)
print("Rows used  :", N_WINDOWS * WINDOW)
print("Tail unused:", len(Xiid) - N_WINDOWS * WINDOW)


# ==========================================================================================
# 5. EXACT RAW MONITOR + PRIMARY SSS
# ==========================================================================================

PREDICTIVE_CONTINUOUS = [
    "binary_confidence",
    "binary_entropy",
    "binary_margin",
    "multiclass_confidence",
    "multiclass_entropy",
    "multiclass_margin",
]

rows = []

print("\n[5] Reproducing 82 clean-IID windows...")

for w_id, idx in enumerate(windows):

    raw = {}

    # ----------------------------------------------------------------------
    # Predictive shifts
    # ----------------------------------------------------------------------

    for c in PREDICTIVE_CONTINUOUS:

        raw[f"{c}_shift"] = (
            standardized_wasserstein(
                ref_signals[c].values,
                iid_signals.iloc[idx][c].values
            )
        )

    # ----------------------------------------------------------------------
    # Predicted-class distributions
    # ----------------------------------------------------------------------

    raw["binary_prediction_js"] = (
        js_divergence(
            ref_signals[
                "binary_predicted_class"
            ].values,

            iid_signals.iloc[idx][
                "binary_predicted_class"
            ].values,

            np.arange(2)
        )
    )

    raw["multiclass_prediction_js"] = (
        js_divergence(
            ref_signals[
                "multiclass_predicted_class"
            ].values,

            iid_signals.iloc[idx][
                "multiclass_predicted_class"
            ].values,

            np.arange(9)
        )
    )

    # ----------------------------------------------------------------------
    # Input-feature distribution
    # ----------------------------------------------------------------------

    feature_d = np.empty(
        68,
        dtype=np.float64
    )

    for j in range(68):

        feature_d[j] = (
            standardized_wasserstein(
                Xcal[:, j],
                Xiid[idx, j]
            )
        )

    raw["feature_shift_mean"] = float(
        np.mean(feature_d)
    )

    raw["feature_shift_median"] = float(
        np.median(feature_d)
    )

    raw["feature_shift_p90"] = float(
        np.quantile(
            feature_d,
            0.90
        )
    )

    raw["feature_shift_max"] = float(
        np.max(feature_d)
    )

    # ----------------------------------------------------------------------
    # Frozen null normalization
    # ----------------------------------------------------------------------

    E = {
        name:
            normalize_indicator(
                name,
                value
            )
        for name, value in raw.items()
    }

    # ----------------------------------------------------------------------
    # Frozen PRIMARY families
    # ----------------------------------------------------------------------

    fam_predictive = float(
        np.median([
            E["binary_confidence_shift"],
            E["binary_entropy_shift"],
            E["binary_margin_shift"],
            E["multiclass_confidence_shift"],
            E["multiclass_entropy_shift"],
            E["multiclass_margin_shift"],
        ])
    )

    fam_predclass = float(
        np.median([
            E["binary_prediction_js"],
            E["multiclass_prediction_js"],
        ])
    )

    fam_feature = float(
        np.median([
            E["feature_shift_mean"],
            E["feature_shift_median"],
            E["feature_shift_p90"],
            E["feature_shift_max"],
        ])
    )

    sss = float(
        np.median([
            fam_predictive,
            fam_predclass,
            fam_feature,
        ])
    )

    if sss <= Q95:
        state = "normal"
    elif sss <= Q99:
        state = "elevated"
    else:
        state = "severe"

    row = {
        "window_id":
            w_id,

        "family__predictive_confidence_uncertainty":
            fam_predictive,

        "family__predicted_class_distribution":
            fam_predclass,

        "family__input_feature_distribution":
            fam_feature,

        "SSS_primary":
            sss,

        "primary_shift_evidence_state":
            state,
    }

    for name, value in raw.items():
        row[f"raw__{name}"] = value

    for name, value in E.items():
        row[f"E__{name}"] = value

    rows.append(row)

    print(
        f"  window {w_id + 1:02d}/{N_WINDOWS}",
        end="\r"
    )


reproduced = pd.DataFrame(rows)

print("\nReproduction calculation: COMPLETE")


# ==========================================================================================
# 6. EXACT REGRESSION AGAINST ACCEPTED V2.6E-4
# ==========================================================================================

print("\n" + "=" * 108)
print("[6] EXACT REGRESSION AGAINST FROZEN V2.6E-4")
print("-" * 108)

common_numeric = [
    c
    for c in reproduced.columns
    if (
        c in frozen.columns
        and c != "window_id"
        and pd.api.types.is_numeric_dtype(
            reproduced[c]
        )
        and pd.api.types.is_numeric_dtype(
            frozen[c]
        )
    )
]

comparison = []

for c in common_numeric:

    diff = np.abs(
        reproduced[c].to_numpy(dtype=float)
        -
        frozen[c].to_numpy(dtype=float)
    )

    comparison.append({
        "column":
            c,

        "max_abs_diff":
            float(np.max(diff)),

        "mean_abs_diff":
            float(np.mean(diff)),
    })

comparison = pd.DataFrame(
    comparison
).sort_values(
    "max_abs_diff",
    ascending=False
)

print(
    comparison.head(30).to_string(
        index=False
    )
)


# ==========================================================================================
# 7. PRIMARY ANCHORS
# ==========================================================================================

sss_diff = np.abs(
    reproduced["SSS_primary"].to_numpy()
    -
    frozen["SSS_primary"].to_numpy()
)

state_match = np.array_equal(
    reproduced[
        "primary_shift_evidence_state"
    ].astype(str).to_numpy(),

    frozen[
        "primary_shift_evidence_state"
    ].astype(str).to_numpy()
)

median_new = float(
    reproduced["SSS_primary"].median()
)

q95_new = float(
    reproduced["SSS_primary"].quantile(.95)
)

counts_new = (
    reproduced[
        "primary_shift_evidence_state"
    ]
    .value_counts()
    .to_dict()
)

print("\nPrimary anchors:")
print("Median reproduced :", median_new)
print("Median frozen     :", 0.15471666155260375)

print("q95 reproduced    :", q95_new)
print("q95 frozen        :", 0.45611921964636687)

print("State counts      :", counts_new)
print("Expected          :", {"normal": 75, "elevated": 5, "severe": 2})

print("Max |Δ SSS|       :", float(sss_diff.max()))
print("States exact      :", state_match)


# ==========================================================================================
# 8. HARD SCIENTIFIC GATE
#
# Previous accepted V2.6E-4:
# - SSS max discrepancy ~7.18e-11
# - feature median raw discrepancy ~1.35e-7 accepted as numerical precision
# ==========================================================================================

median_ok = np.isclose(
    median_new,
    0.15471666155260375,
    atol=1e-10,
    rtol=0
)

q95_ok = np.isclose(
    q95_new,
    0.45611921964636687,
    atol=1e-10,
    rtol=0
)

counts_ok = (
    counts_new.get("normal", 0) == 75
    and counts_new.get("elevated", 0) == 5
    and counts_new.get("severe", 0) == 2
)

sss_ok = (
    float(sss_diff.max())
    <= 1e-9
)

# Raw indicators:
# retain the already accepted feature-median numerical tolerance.
raw_cols = [
    c
    for c in common_numeric
    if c.startswith("raw__")
]

raw_max = {}

for c in raw_cols:

    raw_max[c] = float(
        np.max(
            np.abs(
                reproduced[c].to_numpy(float)
                -
                frozen[c].to_numpy(float)
            )
        )
    )

raw_ok = True

for c, d in raw_max.items():

    tol = (
        5e-7
        if c == "raw__feature_shift_median"
        else 1e-7
    )

    if d > tol:
        raw_ok = False
        print(
            "RAW FAIL:",
            c,
            "diff=",
            d,
            "tol=",
            tol
        )


PASS = (
    median_ok
    and q95_ok
    and counts_ok
    and sss_ok
    and state_match
    and raw_ok
)

print("\n" + "=" * 108)
print("FINAL CLEAN-IID REGRESSION GATE")
print("-" * 108)

print("Median anchor          :", median_ok)
print("q95 anchor             :", q95_ok)
print("State counts 75/5/2    :", counts_ok)
print("State vector exact     :", state_match)
print("SSS max diff <= 1e-9   :", sss_ok)
print("Raw indicators accepted:", raw_ok)

print("\nFINAL REPRODUCTION PASS:", PASS)


if not PASS:
    raise RuntimeError(
        "Frozen clean-IID primary monitor was not reproduced. "
        "STOP before S5."
    )


print("\n" + "=" * 108)
print("V2.8C-R10 COMPLETE — EXACT FROZEN PRIMARY MONITOR RECOVERED")
print("=" * 108)

print("S5 SSS computed         : NO")
print("S5 labels accessed      : NO")
print("S5 outcomes accessed    : NO")
print("Monitor modified        : NO")
print("Normalization modified  : NO")
print("Adaptive policy defined : NO")
print("V1 artifacts modified   : NO")
print("=" * 108)

## A2 — Persist reproducibility evidence (only after PASS)

Save the reproduced table, columnwise numerical comparison, and a machine-readable gate summary to WP2's own folder.

In [ ]:
import json
assert PASS is True, 'Historical R10 gate did not pass; do not save a success report.'
assert len(reproduced)==82 and len(frozen)==82
assert reproduced.window_id.equals(frozen.window_id), 'Window ID mismatch'
reproduced.to_csv(WP2_OUT/'WP2_V8_R10_reproduced_clean_IID.csv',index=False)
comparison.to_csv(WP2_OUT/'WP2_V8_R10_column_differences.csv',index=False)
summary={
 'source_notebook':'02_TrustShift_V2_frozen_pipeline.ipynb',
 'source_cell':'V2.8C-R10',
 'historical_gate_pass':bool(PASS),
 'window_count':int(len(reproduced)),
 'median_sss':median_new,
 'q95_sss':q95_new,
 'state_counts':{k:int(v) for k,v in counts_new.items()},
 'max_abs_sss_diff':float(sss_diff.max()),
 'state_vector_exact':bool(state_match),
 'all_raw_indicators_within_tolerance':bool(raw_ok),
 'q95_state_boundary':float(Q95),
 'q99_state_boundary':float(Q99),
 'target_labels_used':False,
 'deployment_evaluated':False,
 'neural_monitor_reproduced':False,
}
with open(WP2_OUT/'WP2_V8_R10_regression_gate.json','w') as f:json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print('Saved 3 V8 regression artifacts to',WP2_OUT)

## B1 — Detector-specific extension readiness audit

The original R10 has **two predictive streams** (binary and multiclass XGBoost). A neural binary classifier is **not equivalent** to this two-stream monitor. We must specify which confidence/entropy/margin and predicted-class indicators are valid for each neural detector, recalibrate on source-null windows, and compute new source-only SSS boundaries. **Do not reuse XGBoost's numerical boundaries for detector-specific SSS without calibration.** This cell checks the neural inputs and previous WP2 artifacts, without performing a misleading partial replay.

In [ ]:
assert PASS, 'Do not proceed if exact historical regression fails'
from pathlib import Path
import pandas as pd
BASE=Path('/content/drive/MyDrive/TrustShift_IDS')
V2=BASE/'V2_adaptive_reliability'
checks={
 'neural_binary_mlp':BASE/'models',
 'source_calibration':BASE/'data/processed/CICIDS2017_68F_imputed/CICIDS2017_calibration_68F_imputed.parquet',
 'source_reference_signals':V2/'source_reference/V2_calibration_label_free_signals.parquet',
 'frozen_feature_mapping':V2/'source_reference/V2_frozen_68_feature_mapping.csv',
 'V6_raw_feature_audit':WP2_OUT/'WP2_V6_feature_raw_audit.csv',
 'V7_normalized_feature_audit':WP2_OUT/'WP2_V7_feature_normalized_audit.csv',
 'V5_results':WP2_OUT/'WP2_V5_window_results.csv',
}
rows=[{'item':k,'exists':p.exists(),'path':str(p)} for k,p in checks.items()]
manifest=pd.DataFrame(rows)
display(manifest)
manifest.to_csv(WP2_OUT/'WP2_V8_neural_readiness_manifest.csv',index=False)
print('Neural model candidates:')
for p in sorted((BASE/'models').rglob('*')) if (BASE/'models').exists() else []:
 if p.is_file() and any(x in p.name.lower() for x in ('mlp','transformer')):
  print(str(p)[:230])
print('Readiness inventory saved. This is not a neural-monitor validation.')

## B2 — Required protocol before the neural evaluation

1. Reuse exact original standardized Wasserstein feature monitoring and frozen 68-feature mapping.
2. Reconstruct **detector-specific** predictive signals and predicted-class distributions from each neural model's own calibrated probabilities; do not duplicate binary probabilities to masquerade as a multiclass stream.
3. Recompute source-null indicator distributions for each detector using source-only windows. Freeze normalization and SSS boundaries **before** examining deployment outcomes.
4. Independently verify neural clean-IID false-alarm/state behavior. Document all indicator/family adaptations explicitly.
5. Only then run the 4-seed × 60-window deployment comparison with matched window memberships and frozen policies; include transferred-XGBoost states, detector-specific states, no abstention, static Severe. Record selective risk, coverage, deferral and attack outcomes.

**Send back:** `FINAL REPRODUCTION PASS`, state counts, max SSS discrepancy, and B1 manifest. If A1 fails, send the traceback and numerical discrepancy table instead.